In [ ]:
import os 

# set directory to parent directory
os.chdir("..")

# print current working directory
print("Current Working Directory:", os.getcwd())

In [1]:
import numpy as np
import pandas as pd
import torch
from FID.fid_score import calculate_fid, get_precomputed_fid_scores_path, save_fid_stats_as_dict
from matplotlib import pyplot as plt
from models.model import TreeVAE
import yaml
from tqdm import tqdm
from sklearn.metrics.cluster import normalized_mutual_info_score, adjusted_rand_score
from pathlib import Path
from utils.utils import reset_random_seeds, display_image, cluster_acc
from utils.data_utils import get_data, get_gen
from utils.training_utils import compute_leaves, predict, move_to 
from models.model_smalltree import SmallTreeVAE
from models.losses import loss_reconstruction_binary, loss_reconstruction_mse, loss_reconstruction_cov_mse_eval
from utils.model_utils import Node, construct_tree_fromnpy, return_list_tree, construct_data_tree, construct_tree_fromnpy
from utils.model_utils import construct_tree, compute_posterior
from utils.plotting_utils import plot_tree_graph, get_node_embeddings, draw_tree_with_scatter_plots 


# create a graph of the tree
import networkx as nx
import matplotlib.pyplot as plt


In [2]:
base_path = '/cluster/work/vogtlab/Group/jogoncalves/treevae/'

path_1='models/experiments/cubicc/20240923-013355_8ddcc'
path_2='models/experiments/cubicc/20240923-013355_276bd'
path_3='models/experiments/cubicc/20240923-013355_2538a'
path_4='models/experiments/cubicc/20240923-013355_6236b'
path_5='models/experiments/cubicc/20240923-013355_fc06f'
path_6='models/experiments/cubicc/20240923-013355_434df'
path_7='models/experiments/cubicc/20240923-013355_61ff6'
path_8='models/experiments/cubicc/20240923-013355_ac769'
path_9='models/experiments/cubicc/20240923-013340_34724'
path_10='models/experiments/cubicc/20240923-013340_0e296'
vae_path_list=[path_1, path_2, path_3, path_4, path_5, path_6, path_7, path_8, path_9, path_10]


# load data
trainset, trainset_eval, testset = get_data(configs)
gen_train = get_gen(trainset, configs, validation=False, shuffle=False)
gen_train_eval = get_gen(trainset_eval, configs, validation=True, shuffle=False)
gen_test = get_gen(testset, configs, validation=True, shuffle=False)
gen_train_eval_iter = iter(gen_train_eval)
gen_test_iter = iter(gen_test)
y_train = trainset_eval.dataset.targets.clone().detach()[trainset_eval.indices].numpy()
y_test = testset.dataset.targets.clone().detach()[testset.indices].numpy()

['20231114-143553_eec46', '20231115-162449_1d3d3', '20231115-162637_26e44']

# Comparisons:


In [4]:
# if own list of experiments, set to True and add the names of the experiments
# otherwise, set to False to use the list above --> all models in the folder

use_own_list = True

if use_own_list:

    # Shared Decoder vs Separate Decoder:  
    # dir_list =['20231011-165358_f6be0', '20231011-165739_00d85']
    # names = ['Shared Decoder', 'Separate Decoder']

    # Compare size of latent space: 3 vs 4 vs 5 vs 6 latent variables
    #dir_list = ['20231009-091918_37125', '20231009-091849_be1f7', '20231009-091449_7d976', '20231009-091818_59550']
    #names = ['Latent variables: 3', 'Latent variables: 4', 'Latent variables: 5', 'Latent variables: 6']

    # Compare dim of latent space: 4 vs 8 vs 16
    # dir_list = ['20231009-091723_3dec9', '20231009-091449_7d976', '20231009-091748_792e6']
    # names = ['Latent dim: 4', 'Latent dim: 8', 'Latent dim: 16']

    # Compare dim of mlp layers: 64 vs 128 vs 256
    # dir_list = ['20231009-152328_520bc', '20231009-091449_7d976', '20231009-152357_21fde']
    # names = ['MLP layers: 64', 'MLP layers: 128', 'MLP layers: 256']

    # Compare dropout: 0.1 vs 0.5 for 10 epochs in the router 
    # dir_list = ['20231011-190552_3c389', '20231011-180942_6da83']
    # names = ['Dropout: 0.1', 'Dropout: 0.5']

    # Compare normal vs dropout: 0.1 vs dropout :0.3 vs dropout: 0.5 in the router 
    dir_list = ['20231017-155804_0f493', '20231017-155826_6ce4f', '20231017-155852_d9f97', '20231017-155920_7ce06']
    names = ['Normal','Dropout: 0.1', 'Dropout: 0.3', 'Dropout: 0.5']

    # Compare normal vs skip connection in transformations
    # dir_list = ['20231009-091449_7d976', '20231010-173302_0cdb4']
    # names = ['Normal', 'Skip connection']

    # Other experiments
    dir_list = ['20231115-181138_cdba9']
    names = None

else:
    names = None



In [13]:
# prepare a dataframe to store the results
results = pd.DataFrame(columns=['model', 'nmi', 'ari', 'acc', 'data', 'num_clusters_data', 'config_name',
                  'eager_mode', 'seed', 'wandb_logging', 'activation', 'aug_decisions_weight', 'augment',
                    'augmentation_method', 'batch_size', 'compute_ll', 'decay_kl', 'decay_lr', 'decay_stepsize',
                    'encoder', 'grow', 'initial_depth', 'inp_shape', 'intermediate_fulltrain', 'kl_start',
                    'latent_dim', 'lr', 'mlp_layers', 'num_clusters_tree', 'num_epochs', 'num_epochs_finetuning',
                    'num_epochs_intermediate_fulltrain', 'num_epochs_smalltree', 'prune', 'weight_decay'])


for i, model_folder in enumerate(dir_list):
    # skip if directory is empty
    if os.listdir(path + model_folder) == []:
        continue

    if model_folder[0] != '2':
        continue

    checkpoint_path = path + model_folder

    with open(checkpoint_path + "/config.yaml", 'r') as stream:
        configs = yaml.load(stream,Loader=yaml.Loader)

    # model name
    model_name = "ld: " + str(configs['training']['latent_dim']) + ", " "layers: " + str(configs['training']['mlp_layers'])

    print(" ")
    print(" ")
    print(150*"-")

    if names is not None:
        print(names[i])
        print(" ")

    # print info
    print(model_folder)
    print('latent_dim: ', configs['training']['latent_dim'])
    print('mlp_layers: ', configs['training']['mlp_layers'])
    print('num_clusters_tree: ', configs['training']['num_clusters_tree'])
    print('num_epochs: ', configs['training']['num_epochs'])
    print('num_epochs_finetuning: ', configs['training']['num_epochs_finetuning'])
    print('num_epochs_intermediate_fulltrain: ', configs['training']['num_epochs_intermediate_fulltrain'])
    print('num_epochs_smalltree: ', configs['training']['num_epochs_smalltree'])
    
    print(" ")

    # load data
    trainset, trainset_eval, testset = get_data(configs)
    gen_train = get_gen(trainset, configs, validation=False, shuffle=False)
    gen_train_eval = get_gen(trainset_eval, configs, validation=True, shuffle=False)
    gen_test = get_gen(testset, configs, validation=True, shuffle=False)
    gen_train_eval_iter = iter(gen_train_eval)
    gen_test_iter = iter(gen_test)
    y_train = trainset_eval.dataset.targets.clone().detach()[trainset_eval.indices].numpy()
    y_test = testset.dataset.targets.clone().detach()[testset.indices].numpy()


    # load model
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = TreeVAE(**configs['training'])
    data_tree = np.load(checkpoint_path+'/data_tree.npy', allow_pickle=True)

    model = construct_tree_fromnpy(model, data_tree, configs)
    if not (configs['globals']['eager_mode'] and configs['globals']['wandb_logging']!='offline'):
        pass
        #model = torch.compile(model)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model.load_state_dict(torch.load(checkpoint_path+'/model_weights.pt', map_location=device), strict=False)
    model.to(device)
    model.eval()

    # Test Metrics
    n_d = configs['training']['num_clusters_tree']
    prob_leaves = predict(gen_test, model, device,'prob_leaves')
    y = np.squeeze(np.argmax(prob_leaves, axis=-1))
    print(' ')
    print('Test ARI:', adjusted_rand_score(y, np.squeeze(y_test)))
    print('Test NMI:', normalized_mutual_info_score(y, np.squeeze(y_test)))
    print('Test Acc:', cluster_acc(y.numpy(), np.squeeze(y_test)))

    ########################################################################################
    # plot the tree
    plot_tree_graph(data_tree)

    # save the results
    if names is not None:
        name = names[i]
    else:
        name = model_folder


    results.loc[i] = [name, normalized_mutual_info_score(y, np.squeeze(y_test)), adjusted_rand_score(y, np.squeeze(y_test)), cluster_acc(y.numpy(), np.squeeze(y_test)), configs['data']['data_name'], configs['data']['num_clusters_data'], configs['globals']['config_name'],
                    configs['globals']['eager_mode'], configs['globals']['seed'], configs['globals']['wandb_logging'], configs['training']['activation'], configs['training']['aug_decisions_weight'], configs['training']['augment'],
                    configs['training']['augmentation_method'], configs['training']['batch_size'], configs['training']['compute_ll'], configs['training']['decay_kl'], configs['training']['decay_lr'], configs['training']['decay_stepsize'],
                    configs['training']['encoder'], configs['training']['grow'], configs['training']['initial_depth'], configs['training']['inp_shape'], configs['training']['intermediate_fulltrain'], configs['training']['kl_start'],
                    configs['training']['latent_dim'], configs['training']['lr'], configs['training']['mlp_layers'], configs['training']['num_clusters_tree'], configs['training']['num_epochs'], configs['training']['num_epochs_finetuning'],
                    configs['training']['num_epochs_intermediate_fulltrain'], configs['training']['num_epochs_smalltree'], configs['training']['prune'], configs['training']['weight_decay']]
    

    # get node embeddings for each test data point

    # pick data loader
    data_loader = gen_test

    nb_nodes = len(data_tree)

    # each entry in node_embeddings is a dictionary with keys 'prob' and 'z_sample' for each leaf
    node_embeddings = [{'prob': [], 'z_sample': []} for _ in range(nb_nodes)]
    label_list = []

    # iterate over test data points
    for inputs, labels in tqdm(data_loader):
        inputs_gpu, labels_gpu = inputs.to(device), labels.to(device)

        label_list.append(labels)

        with torch.no_grad():
            node_info = get_node_embeddings(model, inputs_gpu)

        # for each node, append the probability and z_sample to the list

        k = 0 # need this variable to skip "no digits" nodes
        for i in range(nb_nodes):
            j = i - k
            if data_tree[i][1] == 'no digits':
                k += 1
                continue

            node_embeddings[i]['prob'].append(node_info[j]['prob'].numpy())
            node_embeddings[i]['z_sample'].append(node_info[j]['z_sample'].numpy())

    # flatten the lists
    k = 0
    for i in range(nb_nodes):
        if data_tree[i][1] == 'no digits':
            node_embeddings[i]['prob'] = []
            node_embeddings[i]['z_sample'] = []
            continue
        
        node_embeddings[i]['prob'] = np.concatenate(node_embeddings[i]['prob'])
        node_embeddings[i]['z_sample'] = np.concatenate(node_embeddings[i]['z_sample'])

    label_list = np.concatenate(label_list)

    # plot the tree with the node embeddings

    # Draw the tree graph with scatter plots as nodes and arrows for edges
    draw_tree_with_scatter_plots(data_tree, node_embeddings, label_list, pca = True)

    ########################################################################################
    # plot generated images

    print(" ")
    print("Generated Images")
    print(" ")


    n_imgs = configs['training']['batch_size']

    with torch.no_grad():
        reconstructions, p_c_z = model.generate_images(n_imgs, device)
    reconstructions = move_to(reconstructions, 'cpu')
    clusterwise_reconst = [torch.zeros_like(reconstructions[0][0:2]) for i in range(len(reconstructions))]
    n_iter=0
    max_iter = 200
    while min([clusterwise_reconst[leaf_ind].shape[0] for leaf_ind in range(len(reconstructions))]) < n_imgs+2 and n_iter < max_iter:
        for i in range(n_imgs):
            leaf_ind = torch.argmax(p_c_z[i])
            if clusterwise_reconst[leaf_ind].shape[0] < n_imgs+2:
                clusterwise_reconst[leaf_ind] = torch.vstack([clusterwise_reconst[leaf_ind], reconstructions[leaf_ind][i].unsqueeze(0)])
        with torch.no_grad():
            reconstructions, p_c_z = model.generate_images(n_imgs, device)
        reconstructions = move_to(reconstructions, 'cpu')
        n_iter += 1

    for i in range(len(reconstructions)):
        clusterwise_reconst[i] = clusterwise_reconst[i][2:,:]

    # show 5 reconstructions per leaf

    n_leaves = len(clusterwise_reconst)
    n_gen = 5
    fig, axs = plt.subplots(n_gen, n_leaves*2, figsize=(15, 5))

    for j in range(n_leaves * 2):
        axs[0, j].set_title(f"Leaf {j//2}")

        for i in range(n_gen):
            if j % 2 == 0:
                axs[i, j].imshow(display_image(clusterwise_reconst[j//2][i]), cmap=plt.get_cmap('gray'))
            else:
                axs[i, j].imshow(display_image(clusterwise_reconst[j//2][i + n_leaves]), cmap=plt.get_cmap('gray'))
            
            axs[i, j].set_axis_off()

    fig.tight_layout()
    plt.show()

KeyError: 'latent_dim'

In [8]:
results

,model,nmi,ari,acc,data,num_clusters_data,config_name,eager_mode,seed,wandb_logging,...,latent_dim,lr,mlp_layers,num_clusters_tree,num_epochs,num_epochs_finetuning,num_epochs_intermediate_fulltrain,num_epochs_smalltree,prune,weight_decay
0,20231022-103943_972f7,0.923109,0.90734,0.9556,mnist,10,mnist,True,11,offline,...,"[8, 8, 8, 8, 8, 8]",0.001,"[128, 128, 128, 128, 128, 128]",10,150,200,80,150,True,0.00001


In [ ]:
results.sort_values(by=['nmi'], ascending=False).head(5)

,model,nmi,ari,acc,data,num_clusters_data,config_name,eager_mode,seed,wandb_logging,...,latent_dim,lr,mlp_layers,num_clusters_tree,num_epochs,num_epochs_finetuning,num_epochs_intermediate_fulltrain,num_epochs_smalltree,prune,weight_decay
1,Dropout: 0.1,0.894598,0.850591,0.9031,mnist,10,mnist,True,42,offline,...,"[8, 8, 8, 8, 8]",0.001,"[128, 128, 128, 128, 128]",10,150,200,80,150,True,0.00001
0,Normal,0.868977,0.830266,0.8768,mnist,10,mnist,True,42,offline,...,"[8, 8, 8, 8, 8]",0.001,"[128, 128, 128, 128, 128]",10,150,200,80,150,True,0.00001
2,Dropout: 0.5,0.861101,0.788967,0.8454,mnist,10,mnist,True,42,offline,...,"[8, 8, 8, 8, 8]",0.001,"[128, 128, 128, 128, 128]",10,150,200,80,150,True,0.00001


#  FID Scores 


In [4]:
model_folder = '20231108-183609_ad0b7'
checkpoint_path = path + model_folder
with open(checkpoint_path + "/config.yaml", 'r') as stream:
    configs = yaml.load(stream,Loader=yaml.Loader)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = TreeVAE(**configs['training'])
data_tree = np.load(checkpoint_path+'/data_tree.npy', allow_pickle=True)

model = construct_tree_fromnpy(model, data_tree, configs)
if not (configs['globals']['eager_mode'] and configs['globals']['wandb_logging']!='offline'):
    pass
    #model = torch.compile(model)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.load_state_dict(torch.load(checkpoint_path+'/model_weights.pt', map_location=device), strict=False)
model.to(device)
model.eval()

TreeVAE(
  (bottom_up): ModuleList(
    (0): EncoderSmallCnn(
      (cnn0): Conv2d(1, 11, kernel_size=(12, 12), stride=(1, 1), padding=(1, 1), bias=False)
      (cnn1): Conv2d(11, 21, kernel_size=(12, 12), stride=(1, 1), padding=(1, 1), bias=False)
      (cnn2): Conv2d(21, 32, kernel_size=(7, 7), stride=(5, 5), padding=(1, 1), bias=False)
      (bn0): BatchNorm2d(11, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (bn1): BatchNorm2d(21, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (bn2): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1-2): 2 x Conv(
      (conv0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn0): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv1): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_r

Generations

In [6]:
n_imgs = 10000

with torch.no_grad():
    generations, p_c_z = model.generate_images(n_imgs, device)

# for each generated image, only save the ones that are in the leaf with the highest probability

generations_list = []

for i in range(n_imgs):  
    # only save generation from leaf with highest probability
    leaf_ind = torch.argmax(p_c_z[i])
    generations_list.append(generations[leaf_ind][i])

# save generated images as torch dataset, empty labels
gen_dataset = torch.stack(generations_list).squeeze()


In [14]:
from FID.fid_score import save_fid_stats_as_dict

stats = save_fid_stats_as_dict(gen_dataset, batch_size=256, device="mps", dims = 2048)

Saving FID statistics


100%|██████████| 40/40 [00:50<00:00,  1.26s/it]


In [16]:
# get FID score

from FID.fid_score import calculate_fid

# get the paths to the real images
#real = 'FID/fid_stats_precomputed/fid_stats_cifar10_train.npz'
real = save_fid_stats_as_dict(testset.dataset.data, batch_size=256, device="mps", dims = 2048)


# calculate the FID score
fid_value = calculate_fid([real, stats], batch_size=256, device="mps", dims=2048)
print("FID score: ", fid_value)

Saving FID statistics


100%|██████████| 40/40 [00:50<00:00,  1.25s/it]


FID score:  238.01689914305615


In [22]:
real = save_fid_stats_as_dict(testset.dataset.data, batch_size=256, device="mps", dims = 2048)

Saving FID statistics


100%|██████████| 40/40 [00:52<00:00,  1.30s/it]


In [23]:
calculate_fid([real, real], batch_size=256, device="mps", dims=2048)

1.1368683772161603e-12

# Save images in folders 


In [8]:
from torchvision.utils import save_image

In [7]:
# save the generated images as png files

from torchvision.utils import save_image

# create a folder to save the images
folder = 'data/generated_images'

# save the images
for i in range(n_imgs):
    save_image(generations_list[i], folder + '/img_' + str(i) + '.png')

NameError: name 'n_imgs' is not defined

In [9]:
# load data
trainset, trainset_eval, testset = get_data(configs)
gen_train = get_gen(trainset, configs, validation=False, shuffle=False)
gen_train_eval = get_gen(trainset_eval, configs, validation=True, shuffle=False)
gen_test = get_gen(testset, configs, validation=True, shuffle=False)
gen_train_eval_iter = iter(gen_train_eval)
gen_test_iter = iter(gen_test)
y_train = trainset_eval.dataset.targets.clone().detach()[trainset_eval.indices].numpy()
y_test = testset.dataset.targets.clone().detach()[testset.indices].numpy()

In [11]:
# save the cifar10 images as png files from testset
folder = 'data/cifar10_images'

# iterate over the testset
for i in range(len(testset)):
    save_image(testset[i][0], folder + '/img_' + str(i) + '.png')